# Домашнее задание №1: Построение сложной сцены из облаков точек

**Выполнил:** evanatas

1. Создать геометрические примитивы (куб, сфера, цилиндр, конус, тор) и облака точек
   (`sample_points_uniformly`, `sample_points_poisson_disk`).
2. Применить трансформации: перенос, масштаб, поворот.
3. Собрать композиции: снеговик, стол, пирамида из кубов, шестилучевая звезда.
4. Визуализировать сцену в Open3D и Plotly.
5. Раскрасить объекты разными цветами.

In [1]:
import open3d as o3d
import numpy as np
import copy
import plotly.graph_objects as go


def center(pcd):
    """Сдвинуть облако точек в начало координат."""
    pcd.translate(-pcd.get_center())
    return pcd

In [2]:
# Задание 1. Примитивы и облака точек через sample_points_uniformly
cube     = o3d.geometry.TriangleMesh.create_box(1.0, 1.0, 1.0)
sphere   = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius=0.5, height=2.0)
cone     = o3d.geometry.TriangleMesh.create_cone(radius=0.5, height=1.0)
torus    = o3d.geometry.TriangleMesh.create_torus(torus_radius=1.0, tube_radius=0.3)

meshes = [cube, sphere, cylinder, cone, torus]
colors = [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [0,1,1]]

uniform = []
for k, (m, c) in enumerate(zip(meshes, colors)):
    pcd = m.sample_points_uniformly(number_of_points=1000)
    pcd.paint_uniform_color(c)
    pcd.translate((k * 3, 0, 0))      # разносим в ряд, чтобы не накладывались
    uniform.append(pcd)

o3d.visualization.draw_geometries(uniform, window_name="Примитивы (uniformly)")

In [3]:
# Задание 1. Те же примитивы через sample_points_poisson_disk (равномернее)
poisson = []
for k, (m, c) in enumerate(zip(meshes, colors)):
    pcd = m.sample_points_poisson_disk(number_of_points=800)
    pcd.paint_uniform_color(c)
    pcd.translate((k * 3, 0, 0))
    poisson.append(pcd)

o3d.visualization.draw_geometries(poisson, window_name="Примитивы (poisson disk)")

In [4]:
# Задание 2. Трансформации: перенос, масштаб, поворот
base = [m.sample_points_uniformly(1000) for m in meshes]
for p, c in zip(base, colors):
    p.paint_uniform_color(c)

# перенос куба
cube_t = copy.deepcopy(base[0]).translate((-4, 0, 0))
# сфера без изменений
sphere_t = copy.deepcopy(base[1])
# масштаб цилиндра (в 1.8 раза)
cyl_t = copy.deepcopy(base[2]).translate((4, 0, 0))
cyl_t.scale(1.8, center=cyl_t.get_center())
# поворот конуса на 90 град вокруг X
cone_t = copy.deepcopy(base[3]).translate((0, 4, 0))
cone_t.rotate(cone_t.get_rotation_matrix_from_xyz((np.pi/2, 0, 0)), center=cone_t.get_center())
# поворот тора на 90 град вокруг X
torus_t = copy.deepcopy(base[4]).translate((0, -4, 0))
torus_t.rotate(torus_t.get_rotation_matrix_from_xyz((np.pi/2, 0, 0)), center=torus_t.get_center())

o3d.visualization.draw_geometries([cube_t, sphere_t, cyl_t, cone_t, torus_t],
                                  window_name="Трансформации")

In [5]:
# Задание 3. Снеговик из трёх сфер + конус-нос
def sphere_pcd(radius, n, color, z):
    p = o3d.geometry.TriangleMesh.create_sphere(radius).sample_points_uniformly(n)
    p.paint_uniform_color(color)
    p.translate((0, 0, z))
    return p

bottom = sphere_pcd(1.0, 1500, [0.9, 0.9, 1.0], 1.0)
middle = sphere_pcd(0.7, 1000, [0.9, 0.9, 1.0], 2.4)
head   = sphere_pcd(0.4,  700, [0.9, 0.9, 1.0], 3.3)

nose = o3d.geometry.TriangleMesh.create_cone(0.1, 0.4).sample_points_uniformly(250)
nose.paint_uniform_color([1.0, 0.5, 0.0])
nose.rotate(nose.get_rotation_matrix_from_xyz((-np.pi/2, 0, 0)), center=(0, 0, 0))  # нос вперёд (+Y)
nose.translate((0, 0.4, 3.3))

snowman = bottom + middle + head + nose
o3d.visualization.draw_geometries([snowman], window_name="Снеговик")

In [6]:
# Задание 3. Стол: столешница (куб) + 4 ножки (цилиндры)
top = center(o3d.geometry.TriangleMesh.create_box(3.0, 2.0, 0.2).sample_points_uniformly(2000))
top.translate((0, 0, 1.6))
top.paint_uniform_color([0.5, 0.3, 0.1])

leg = center(o3d.geometry.TriangleMesh.create_cylinder(0.1, 1.5).sample_points_uniformly(500))
leg.paint_uniform_color([0.4, 0.2, 0.0])

table = top
for sx, sy in [(1.3, 0.8), (-1.3, 0.8), (1.3, -0.8), (-1.3, -0.8)]:
    table += copy.deepcopy(leg).translate((sx, sy, 0.75))

o3d.visualization.draw_geometries([table], window_name="Стол")

In [7]:
# Задание 3. Пирамида из кубов (ярусы 3x3, 2x2, 1x1)
base_cube = center(o3d.geometry.TriangleMesh.create_box(1.0, 1.0, 1.0).sample_points_uniformly(500))
layer_colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]

pyramid = o3d.geometry.PointCloud()
for level, side in enumerate([3, 2, 1]):
    z = 0.5 + level
    for i in range(side):
        for j in range(side):
            x = (i - (side - 1) / 2)
            y = (j - (side - 1) / 2)
            cube = copy.deepcopy(base_cube).translate((x, y, z))
            cube.paint_uniform_color(layer_colors[level])
            pyramid += cube

o3d.visualization.draw_geometries([pyramid], window_name="Пирамида из кубов")

In [8]:
# Задание 3. Шестилучевая звезда из вытянутых прямоугольников
beam = center(o3d.geometry.TriangleMesh.create_box(3.0, 0.5, 0.5).sample_points_uniformly(1000))
beam.translate((1.5, 0, 0))             # один конец в центре
beam.paint_uniform_color([0.9, 0.9, 0.0])

star = o3d.geometry.PointCloud()
for i in range(6):
    R = beam.get_rotation_matrix_from_xyz((0, 0, np.deg2rad(i * 60)))
    star += copy.deepcopy(beam).rotate(R, center=(0, 0, 0))

o3d.visualization.draw_geometries([star], window_name="Шестилучевая звезда")

In [9]:
# Задание 4. Итоговая сцена: расставляем композиции по углам + пол
scene_snowman = copy.deepcopy(snowman).translate((-7, 7, 0))
scene_table   = copy.deepcopy(table).translate((7, 7, 0))
scene_pyramid = copy.deepcopy(pyramid).translate((-7, -7, 0))
scene_star    = copy.deepcopy(star).translate((7, -7, 0))
scene_star.scale(0.6, center=scene_star.get_center())

floor = center(o3d.geometry.TriangleMesh.create_box(25, 25, 0.1).sample_points_uniformly(8000))
floor.paint_uniform_color([0.5, 0.5, 0.5])

final_scene = scene_snowman + scene_table + scene_pyramid + scene_star + floor
o3d.visualization.draw_geometries([final_scene], window_name="Итоговая сцена")

In [10]:
# Задание 4. Та же сцена в Plotly + сохранение картинки в results/results.png
pts = np.asarray(final_scene.points)
cols = np.asarray(final_scene.colors)

fig = go.Figure(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode="markers",
    marker=dict(size=2, color=cols),
))
fig.update_layout(
    title_text="Итоговая сцена (Plotly)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    margin=dict(l=0, r=0, b=0, t=40),
)

import os
os.makedirs("results", exist_ok=True)
fig.write_image("results/results.png", width=1100, height=800, scale=2)
fig.show()